In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import jax
from jax.scipy.special import bessel_jn
from notebooks.fft import *

# Note: currently only necessary for calculating the Airy from its simple analytic form
# Immediately working in the corresponding frequeny domain of the spot, with a circle corresponding to the NA,
# would avoid this problem
jax.config.update("jax_enable_x64", True)
jax.devices()

In [ ]:
from params import *

N = 1024

In [ ]:
W = p
H = p
plot_unit_l = unit_l * 1e6  # um
r_extent = [
    -W / 2 * plot_unit_l,
    W / 2 * plot_unit_l,
    -H / 2 * plot_unit_l,
    H / 2 * plot_unit_l,
]
X, Y, R = create_axes(W, H, 1, N=N)

In [ ]:
# Airy disk
spot_R = k_0 * NA * jnp.abs(R)
E_airy = 2 * bessel_jn(spot_R + eps, v=1)[1, :] / (spot_R + eps)

In [ ]:
plt.imshow(E_airy, extent=r_extent)
plt.xlabel(r"$x$ [um]")
plt.ylabel(r"$y$ [um]")
plt.show()
plt.imshow(E_airy**2, extent=r_extent)
plt.xlabel(r"$x$ [um]")
plt.ylabel(r"$y$ [um]")

In [ ]:
F_airy = fft2_shifted(E_airy)
f_x, f_y = ft_image_coordinates(E_airy, W, H)
k_x = 2 * jnp.pi * f_x
k_y = 2 * jnp.pi * f_y
k = jnp.sqrt(k_x**2 + k_y**2)
k_filter = jnp.astype(k <= k_0, jnp.float64)

In [ ]:
from matplotlib.patches import Circle

fig, ax = plt.subplots()
k_min = jnp.min(k_x)
k_max = jnp.max(k_x)

k_extent = [k_min, k_max, k_min, k_max]
my_circle = Circle((0, 0), k_0 * NA, linestyle="--", color="red", fill=False)

ax.add_patch(my_circle)
plt.imshow(jnp.abs(F_airy), extent=k_extent)
plt.xlim((-k_0, k_0))
plt.ylim((-k_0, k_0))

## Back propagating the Airy spot

In [ ]:
# From now on, 32-bit floats are enough
# jax.config.update("jax_enable_x64", False)

### Constructing the ideal circular spectrum

In [ ]:
F_airy = jnp.astype(k <= k_0 * NA, jnp.float32)
fig, ax = plt.subplots()
k_min = jnp.min(k_x)
k_max = jnp.max(k_x)

k_extent = [k_min, k_max, k_min, k_max]
my_circle = Circle((0, 0), k_0 * NA, linestyle="--", color="red", fill=False)

ax.add_patch(my_circle)
plt.imshow(jnp.abs(F_airy), extent=k_extent)
plt.xlim((-k_0, k_0))
plt.ylim((-k_0, k_0))

In [ ]:
phi_prop = d * jnp.sqrt(k_0**2 - k_filter * k**2 + eps) + d_cov * jnp.sqrt(
    (n_cov * k_0) ** 2 - k_filter * k**2 + eps
)
plt.xlim((-k_0, k_0))
plt.ylim((-k_0, k_0))
plt.imshow(phi_prop, extent=k_extent)

In [ ]:
F_exit = jnp.exp(-1j * phi_prop) * F_airy * k_filter
E_exit = ifft2_shifted(F_exit)
plt.imshow(jnp.abs(E_exit), extent=r_extent, cmap="hot")
plt.xlabel(r"$x$ [um]")
plt.ylabel(r"$y$ [um]")

In [ ]:
arg_phase = jnp.angle(E_exit)
plt.imshow(arg_phase, extent=r_extent, cmap="gist_rainbow")
plt.xlabel(r"$x$ [um]")
plt.ylabel(r"$y$ [um]")

In [ ]:
phase_round_off = -0.45833 * jnp.pi


def calc_power(x):
    return jnp.sum(jnp.abs(x) ** 2)


def make_phi_design(arg_phase, phase_round_off):
    phi_rounded = jnp.floor((arg_phase + phase_round_off) / jnp.pi + 1.5)
    phi_design = jnp.pi * jnp.mod(phi_rounded, 2)
    return phi_design


phi_design = make_phi_design(arg_phase, phase_round_off)

T_design = 1 / N * jnp.exp(1j * phi_design)
P = calc_power(T_design)
plt.imshow(jnp.angle(-T_design), extent=r_extent, cmap="binary"), P

## Forward pass of binarized design

In [ ]:
T_spectrum = fft2_shifted(T_design)
fig, ax = plt.subplots()
my_circle = Circle((0, 0), k_0 * NA, linestyle="--", color="red", fill=False)

ax.add_patch(my_circle)
plt.imshow(jnp.abs(T_spectrum), extent=k_extent)
plt.xlim((-k_0, k_0))
plt.ylim((-k_0, k_0))

In [ ]:
F_im = jnp.exp(1j * phi_prop) * T_spectrum * k_filter

In [ ]:
E_im = ifft2_shifted(F_im)
plt.imshow(jnp.abs(E_im), extent=r_extent)
plt.show()
plt.imshow(jnp.abs(E_im) ** 2, extent=r_extent)

## Quality factor $Q$ calculation

In [ ]:
R_airy = 0.61 * lambda_0 / (NA * unit_l)
R_mask = jnp.astype(R <= R_airy, jnp.float32)


@jit
def calculate_Q_relative(pattern):
    total_P = calc_power(pattern)
    Q_p = calc_power(pattern * R_mask)
    return Q_p / total_P


Q_airy = calculate_Q_relative(E_airy)
calculate_Q_relative(E_im), Q_airy

## Generate movie by varying $\epsilon$

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import jax.numpy as jnp

dpi = 300
bitrate = 7200
fps = 30
frames = 300

fig, ax = plt.subplots(dpi=dpi)

ax.set_xlabel(r"$x$ [um]")
ax.set_ylabel(r"$y$ [um]")

# ims is a list of lists, each row is a list of artists to draw in the current frame
# both the image and the epsilon text are animated
ims = []

round_offs = jnp.linspace(-jnp.pi / 2, jnp.pi / 2, frames)

for i, phase_round_off in enumerate(round_offs):
    design = make_phi_design(arg_phase, phase_round_off)

    # image artist for this frame
    im = ax.imshow(design, extent=r_extent, animated=True, cmap="binary")

    # text artist for this frame (axes coordinates: 0..1)
    txt = ax.text(
        0.02,
        0.98,
        rf"$\epsilon$ = {float(phase_round_off/jnp.pi):.4f}",
        transform=ax.transAxes,
        va="top",
        ha="left",
        animated=True,
        bbox=dict(facecolor="white", alpha=1.0, edgecolor="none"),
    )

    if i == 0:
        # optional: show initial frame (static) before animation starts
        ax.imshow(design, extent=r_extent, cmap="binary")
        ax.text(
            0.02,
            0.98,
            rf"$\epsilon$ = {float(phase_round_off/jnp.pi):.4f}",
            transform=ax.transAxes,
            va="top",
            ha="left",
            bbox=dict(facecolor="white", alpha=1.0, edgecolor="none"),
        )

    ims.append([im, txt])

ani = animation.ArtistAnimation(fig, ims, interval=50, blit=True, repeat_delay=1000)
writer = animation.FFMpegWriter(fps=fps, metadata=dict(artist="Me"), bitrate=bitrate)
ani.save("animation-epsilon.mp4", writer=writer)